# Lab 2 Curve Fitting

### Fit data for speaker lab

# Lab 2: Speaker Characterization Analysis

## Part 1: Effective Mass and Stiffness Determination

From theory: $T^2 = \frac{4\pi^2}{s}(m_0 + m_1)$

Where:
- $T$ = period (1/frequency)
- $s$ = effective spring constant
- $m_0$ = effective mass of speaker
- $m_1$ = added mass

We'll determine $m_0$ and $s$ from the slope and intercept of $T^2$ vs added mass.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

# ============================================================================
# LOAD DATA FOR ALL THREE CONFIGURATIONS
# ============================================================================

# Load data
data_no_box = pd.read_csv('LAB_2_Without_Box.csv')
data_box = pd.read_csv('LAB_2_Box_No_Stuff.csv')
data_box_stuff = pd.read_csv('LAB_2_Box_With_Stuff.csv')

print("="*70)
print("LAB 2: SPEAKER CHARACTERIZATION - EFFECTIVE MASS & STIFFNESS")
print("="*70)

# Calculate period (T = 1/f) and period squared for all configurations
for name, data in [('Without Box', data_no_box), ('Box (No Stuff)', data_box), ('Box (With Stuff)', data_box_stuff)]:
    data['period_s'] = 1.0 / data['frequency_Hz']
    data['period_squared_s2'] = data['period_s']**2
    data['mass_kg'] = data['mass_g'] / 1000.0  # Convert to kg
    
    print(f"\n{name}:")
    print(data[['mass_g', 'frequency_Hz', 'period_s', 'period_squared_s2']].to_string(index=False))

# ============================================================================
# LINEAR REGRESSION: T² vs Mass
# ============================================================================

def analyze_speaker_config(data, config_name, color):
    """Perform linear fit and extract m0 and s"""
    
    # Linear fit: T² = slope * m + intercept
    # where slope = 4π²/s and intercept = 4π²m₀/s
    slope, intercept, r_value, p_value, std_err = stats.linregress(
        data['mass_kg'], data['period_squared_s2'])
    
    # Calculate spring constant: s = 4π²/slope
    s = 4 * np.pi**2 / slope
    s_uncertainty = (4 * np.pi**2 / slope**2) * std_err
    
    # Calculate effective mass: m₀ = intercept * s / (4π²)
    m0 = intercept * s / (4 * np.pi**2)
    # Uncertainty propagation for m0 is complex, use simplified estimate
    m0_uncertainty = m0 * 0.05  # ~5% estimate
    
    # Resonance frequency at zero mass
    f0 = 1.0 / np.sqrt(intercept)
    
    return {
        'config': config_name,
        'slope': slope,
        'intercept': intercept,
        'r_squared': r_value**2,
        's': s,
        's_uncertainty': s_uncertainty,
        'm0': m0,
        'm0_uncertainty': m0_uncertainty,
        'f0': f0,
        'color': color,
        'std_err': std_err
    }

# Analyze all three configurations
results = []
results.append(analyze_speaker_config(data_no_box, 'Without Box', 'blue'))
results.append(analyze_speaker_config(data_box, 'Box (No Stuff)', 'red'))
results.append(analyze_speaker_config(data_box_stuff, 'Box (With Stuff)', 'green'))

# Print results
print("\n" + "="*70)
print("ANALYSIS RESULTS:")
print("="*70)
for res in results:
    print(f"\n{res['config']}:")
    print(f"  Slope:              {res['slope']:.6f} ± {res['std_err']:.6f} s²/kg")
    print(f"  Intercept:          {res['intercept']:.6f} s²")
    print(f"  R²:                 {res['r_squared']:.6f}")
    print(f"  Spring constant s:  {res['s']:.2f} ± {res['s_uncertainty']:.2f} N/m")
    print(f"  Effective mass m₀:  {res['m0']*1000:.2f} ± {res['m0_uncertainty']*1000:.2f} g")
    print(f"  Resonance freq f₀:  {res['f0']:.2f} Hz")

In [ ]:
# ============================================================================
# PLOT: T² vs Added Mass (All Configurations)
# ============================================================================

plt.figure(figsize=(14, 9))

# Plot data and fits for all three configurations
datasets = [(data_no_box, 'Without Box'), (data_box, 'Box (No Stuff)'), (data_box_stuff, 'Box (With Stuff)')]

for (data, name), res in zip(datasets, results):
    # Plot experimental data
    plt.plot(data['mass_kg'] * 1000, data['period_squared_s2'], 
             'o', markersize=10, color=res['color'], 
             label=f"{name} - Data", zorder=3)
    
    # Plot best fit line
    mass_fit = np.linspace(0, data['mass_kg'].max() * 1.1, 100)
    T2_fit = res['slope'] * mass_fit + res['intercept']
    plt.plot(mass_fit * 1000, T2_fit, '--', linewidth=2.5, color=res['color'],
             label=f"{name} - Fit: $m_0$ = {res['m0']*1000:.1f} g, $s$ = {res['s']:.1f} N/m\n" + 
                   f"$R^2$ = {res['r_squared']:.5f}",
             zorder=2)

# Formatting
plt.grid(True, alpha=0.3, linestyle='--')
plt.xlabel('Added Mass (g)', fontsize=14, fontweight='bold')
plt.ylabel('Period Squared, T² (s²)', fontsize=14, fontweight='bold')
plt.title('Speaker Characterization: Period² vs Added Mass\nDetermining Effective Mass and Spring Constant', 
          fontsize=16, fontweight='bold', pad=20)
plt.legend(fontsize=10, loc='upper left', framealpha=0.95)
plt.xlim(-2, data_box['mass_g'].max() + 5)

plt.tight_layout()
plt.show()

# ============================================================================
# COMPARISON TABLE
# ============================================================================

print("\n" + "="*70)
print("COMPARISON OF CONFIGURATIONS:")
print("="*70)
comparison_df = pd.DataFrame({
    'Configuration': [r['config'] for r in results],
    'f₀ (Hz)': [f"{r['f0']:.2f}" for r in results],
    'm₀ (g)': [f"{r['m0']*1000:.2f} ± {r['m0_uncertainty']*1000:.2f}" for r in results],
    's (N/m)': [f"{r['s']:.2f} ± {r['s_uncertainty']:.2f}" for r in results],
    'R²': [f"{r['r_squared']:.6f}" for r in results]
})
print(comparison_df.to_string(index=False))

print("\n" + "="*70)
print("PHYSICAL INTERPRETATION:")
print("="*70)
print(f"• Without box: f₀ = {results[0]['f0']:.1f} Hz (lowest resonance)")
print(f"• With box (no stuff): f₀ = {results[1]['f0']:.1f} Hz (highest resonance)")
print(f"• With box + stuff: f₀ = {results[2]['f0']:.1f} Hz (intermediate)")
print(f"\nBox effect: Increases stiffness by {(results[1]['s']/results[0]['s'] - 1)*100:.1f}%")
print(f"Stuffing effect: Reduces stiffness by {(1 - results[2]['s']/results[1]['s'])*100:.1f}% vs empty box")
print("="*70)

## Part 2: Mechanical Resistance Determination

From theory: $\ln(|V_n|) = \ln(V_0) - \beta t_n$

Where:
- $\beta = R_m / (2m_0)$ = damping coefficient
- $R_m$ = mechanical resistance
- $m_0$ = effective mass from Part 1

The slope of $\ln(|V|)$ vs time equals $-\beta$.

In [ ]:
# ============================================================================
# MECHANICAL RESISTANCE ANALYSIS
# ============================================================================

# Load mechanical resistance data
data_decay = pd.read_csv('LAB_2_Mechanical_Resistance.csv')

print("\n" + "="*70)
print("LAB 2: MECHANICAL RESISTANCE FROM FREE DECAY")
print("="*70)
print("\nFree Decay Data:")
print(data_decay.to_string(index=False))

# Perform linear regression: ln(|V|) vs time
# Note: The ln_abs_V column already has the natural log calculated
slope_decay, intercept_decay, r_value_decay, p_value_decay, std_err_decay = stats.linregress(
    data_decay['time_s'], data_decay['ln_abs_V'])

# Calculate damping coefficient: β = -slope (negative because voltage decays)
beta = -slope_decay
beta_uncertainty = std_err_decay

# Use m0 from "Without Box" configuration (most common measurement)
m0_speaker = results[0]['m0']  # in kg

# Calculate mechanical resistance: Rm = 2 * β * m0
R_m = 2 * beta * m0_speaker
R_m_uncertainty = 2 * beta_uncertainty * m0_speaker

# Calculate time constant: τ = 1/β
tau = 1.0 / beta
tau_uncertainty = tau * (beta_uncertainty / beta)

# Calculate quality factor: Q = ω₀/(2β) where ω₀ = 2πf₀
omega_0 = 2 * np.pi * results[0]['f0']
Q = omega_0 / (2 * beta)

print("\n" + "="*70)
print("MECHANICAL RESISTANCE RESULTS:")
print("="*70)
print(f"Linear fit: ln(|V|) = {intercept_decay:.3f} - {beta:.2f}*t")
print(f"R² (goodness of fit):       {r_value_decay**2:.6f}")
print(f"\nDamping coefficient β:       {beta:.2f} ± {beta_uncertainty:.2f} s⁻¹")
print(f"Time constant τ:             {tau*1000:.2f} ± {tau_uncertainty*1000:.2f} ms")
print(f"Mechanical resistance R_m:   {R_m:.4f} ± {R_m_uncertainty:.4f} kg/s")
print(f"Quality factor Q:            {Q:.2f}")
print(f"\nUsing m₀ = {m0_speaker*1000:.2f} g (from 'Without Box' measurement)")
print("="*70)

In [ ]:
# ============================================================================
# PLOT: MECHANICAL RESISTANCE - ln(|V|) vs Time
# ============================================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# LEFT PLOT: Voltage decay (not in log scale)
# Calculate actual voltage from ln(|V|)
V_actual = np.exp(data_decay['ln_abs_V'])
V_fit_line = np.exp(intercept_decay) * np.exp(slope_decay * data_decay['time_s'])

ax1.plot(data_decay['time_s'] * 1000, data_decay['voltage_V'], 
         'bo', markersize=8, label='Experimental Data (Peaks & Troughs)', zorder=3)
ax1.plot(data_decay['time_s'] * 1000, V_fit_line, 
         'r-', linewidth=2, label=f'Exponential Envelope: $V_0 e^{{-\\beta t}}$\n' + 
         f'$\\beta$ = {beta:.2f} s⁻¹, $\\tau$ = {tau*1000:.1f} ms', zorder=2)
ax1.plot(data_decay['time_s'] * 1000, -V_fit_line, 
         'r-', linewidth=2, zorder=2)  # Negative envelope

ax1.set_xlabel('Time (ms)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Voltage (V)', fontsize=12, fontweight='bold')
ax1.set_title('Free Decay Oscillations\nVoltage vs Time', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=10)

# RIGHT PLOT: ln(|V|) vs time (linear relationship)
time_fit = np.linspace(0, data_decay['time_s'].max() * 1.1, 100)
ln_V_fit = intercept_decay + slope_decay * time_fit

ax2.plot(data_decay['time_s'] * 1000, data_decay['ln_abs_V'], 
         'go', markersize=10, label='Experimental Data', zorder=3)
ax2.plot(time_fit * 1000, ln_V_fit, 
         'r--', linewidth=2.5, 
         label=f'Linear Fit: $\\ln(|V|)$ = {intercept_decay:.3f} - {abs(slope_decay):.2f}t\n' +
         f'$R^2$ = {r_value_decay**2:.6f}', zorder=2)

ax2.set_xlabel('Time (ms)', fontsize=12, fontweight='bold')
ax2.set_ylabel('ln(|V|) (dimensionless)', fontsize=12, fontweight='bold')
ax2.set_title('Logarithmic Decay Analysis\nln(|Voltage|) vs Time', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "="*70)
print("LAB 2: COMPLETE SPEAKER CHARACTERIZATION SUMMARY")
print("="*70)
print("\nPART 1: Effective Mass and Spring Constant")
print("-" * 70)
for res in results:
    print(f"\n{res['config']}:")
    print(f"  Resonance frequency f₀:  {res['f0']:.2f} Hz")
    print(f"  Effective mass m₀:       {res['m0']*1000:.2f} g")
    print(f"  Spring constant s:       {res['s']:.2f} N/m")

print("\n" + "-" * 70)
print("\nPART 2: Mechanical Damping (Without Box)")
print("-" * 70)
print(f"  Damping coefficient β:   {beta:.2f} s⁻¹")
print(f"  Mechanical resistance:   {R_m:.4f} kg/s")
print(f"  Time constant τ:         {tau*1000:.1f} ms")
print(f"  Quality factor Q:        {Q:.2f}")

print("\n" + "="*70)
print("PHYSICAL INSIGHTS:")
print("="*70)
print("1. Box Effect on Resonance:")
print(f"   - Adding a box increases resonance from {results[0]['f0']:.1f} Hz to {results[1]['f0']:.1f} Hz")
print(f"   - This is due to air spring inside the box adding stiffness")
print("\n2. Stuffing Effect:")
print(f"   - Stuffing reduces resonance to {results[2]['f0']:.1f} Hz")
print(f"   - Absorbs sound energy, effectively softening the air spring")
print("\n3. Damping:")
print(f"   - Time constant of {tau*1000:.1f} ms means amplitude decays to 37% in this time")
print(f"   - Q-factor of {Q:.1f} indicates {'under' if Q < 0.5 else 'over' if Q > 2 else 'critical'}damped system")
print("="*70)